# GI findings → Excel (report-team handoff)

1. Pick a **client** (GI) — `data/clients/{client}/gi/rules.md`
2. Optionally **rebuild** checkpoints + checkspecs from `rules.md` (harness or legacy ` ```check ` format)
3. Pick a **folder of report JSON** (or leave empty for that client's `corrected/`)
4. Run production PolicyGuard (`run_production_review`)
5. Write Excel under `data/pipeline/findings/`

**Harness `rules.md` format** (Ribkoff): blocks with `**id:**` / `**check:**` / `**where:**` / `**action:**` / `**param:**` / `**when:**`.  
Pipeline maps Action+Where → operators via `src/harness_rules.py` (same as Excel dropdowns in `data/library/`).

Team-facing columns (kept short on purpose):

| Column | Violations | Unable (photo/attachment **content** only) |
|---|---|---|
| What was checked | `GI Instructions` | `GI Instructions - Unable to check` |
| Checkpoint | Where locus (e.g. `Carton Drop Test`) | `Photo: …` or `Attachment: <filename>` |
| Original Content | Exact report value extracted | Photo caption / attachment filename |
| Non-confirmities | `Rules: … Inconsistency: …` | Exact **What to check** text |
| Suggested actions | Verb-first fix | Manual check — processing not supported yet |

Out-of-report rules are **not** written. **Finding Verdict** / **Remark** stay blank for the report team.

In [1]:
from __future__ import annotations

import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from clients import ensure_pipeline_dirs, list_clients, load_client
from findings_export import (
    checkpoint_meta_by_id,
    order_id_for_report,
    requirements_by_id,
    write_findings_workbook,
)
from gi_review import format_cost_summary, load_checkpoints
from review import run_production_review

os.environ.setdefault("GI_PIPELINE_LOG", "0")

# =============================================================================
# CONFIG — edit these
# =============================================================================
CLIENT = "ribkoff"  # must be in list_clients() below

# Rebuild checkpoints + checkspecs from gi/rules.md before review.
# Supports harness format (**id:**/**action:**/**where:**) and legacy ```check fences.
REBUILD_FROM_RULES = True

# Folder of report *.json to review. Examples:
#   "data/clients/July17_Hallamark"
#   "data/clients/ribkoff/corrected"
#   "data/clients/ribkoff/flawed"
#   "/absolute/path/to/my_batch"
# Leave empty ("") to use that client's corrected/ folder.
REPORTS_DIR = "data/clients/ribkoff/flawed"

# Optional: only keep filenames matching this glob (inside REPORTS_DIR).
# Examples: "Q*.json", "*.json", "Q2617*.json"
REPORT_GLOB = "Q*.json"

# If True, ignore REPORTS_DIR and run the client's labeled flawed/ reports.
USE_FLAWED = False

ENABLE_VISION = False
EXTRACT_MODEL = "gemini-3.1-flash-lite"  # atoms + judge; vision uses GI_VISION_MODEL or same
OUT_DIR = PROJECT_ROOT / "data/pipeline/findings"
# =============================================================================

SKIP_NAME_FRAGMENTS = (
    "crawl_summary",
    "pdf_download_summary",
    "pdf_to_json_summary",
    "injection_manifest",
)


def resolve_path(raw: str | Path) -> Path:
    p = Path(raw).expanduser()
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    return p.resolve()


def list_report_jsons(folder: Path, glob_pat: str = "*.json") -> list[Path]:
    """Pick report JSON files from a folder (skip summaries / nested md dumps)."""
    if not folder.is_dir():
        raise FileNotFoundError(f"REPORTS_DIR not found: {folder}")
    out: list[Path] = []
    for path in sorted(folder.glob(glob_pat)):
        if not path.is_file() or path.suffix.lower() != ".json":
            continue
        name = path.name.lower()
        if any(frag in name for frag in SKIP_NAME_FRAGMENTS):
            continue
        out.append(path)
    return out


ensure_pipeline_dirs(PROJECT_ROOT)
OUT_DIR.mkdir(parents=True, exist_ok=True)

available = list_clients(PROJECT_ROOT)
print("clients:", available)
print("CLIENT:", CLIENT)
print("REBUILD_FROM_RULES:", REBUILD_FROM_RULES)
print("REPORTS_DIR:", REPORTS_DIR or f"(client corrected/)")
print("REPORT_GLOB:", REPORT_GLOB)
print("USE_FLAWED:", USE_FLAWED)
print("ENABLE_VISION:", ENABLE_VISION)
print("OUT_DIR:", OUT_DIR)


clients: ['cemaco', 'dfi', 'hallmark', 'new_era', 'ribkoff', 'tpw']
CLIENT: ribkoff
REBUILD_FROM_RULES: True
REPORTS_DIR: data/clients/ribkoff/flawed
REPORT_GLOB: Q*.json
USE_FLAWED: False
ENABLE_VISION: False
OUT_DIR: /Users/williambrun/Documents/test_GI_reports/data/pipeline/findings


/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other 

In [2]:
if CLIENT not in available:
    raise ValueError(f"Unknown CLIENT={CLIENT!r}. Choose one of: {available}")

client = load_client(CLIENT, PROJECT_ROOT)

# ---------------------------------------------------------------------------
# Optional: rebuild checkpoints + checkspecs from gi/rules.md
# (harness **id:**/**action:**/**where:** or legacy ```check fences)
# ---------------------------------------------------------------------------
if REBUILD_FROM_RULES:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
    from build_checkpoints import build_checkpoints
    from compiler import compile_gi

    if not client.gi_rules.exists():
        raise FileNotFoundError(f"Missing GI rules: {client.gi_rules}")
    payload = build_checkpoints(client.gi_rules)
    client.checkpoints_path.parent.mkdir(parents=True, exist_ok=True)
    client.checkpoints_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    sample = client.corrected_reports()
    sample_path = sample[0] if sample else None
    specs = compile_gi(
        client.checkpoints_path,
        client.checkspecs_path,
        sample_report_path=sample_path,
    )
    print(
        f"rebuilt from {client.gi_rules.relative_to(PROJECT_ROOT)} → "
        f"{payload['meta']['checkpoint_count']} checkpoints "
        f"(format={payload['meta'].get('format')}) → {len(specs)} specs"
    )

if not client.checkpoints_path.exists():
    raise FileNotFoundError(
        f"Missing checkpoints: {client.checkpoints_path}\n"
        "Set REBUILD_FROM_RULES=True or run:\n"
        "  python scripts/build_checkpoints.py data/clients/<client>/gi/rules.md "
        "-o data/pipeline/checkpoints/<client>_checkpoints.json\n"
        "  python scripts/compile_gi.py data/pipeline/checkpoints/<client>_checkpoints.json"
    )

from obligation import load_checkspecs

checkpoints = load_checkpoints(client.checkpoints_path)
requirements = requirements_by_id(checkpoints)
meta_by_id = checkpoint_meta_by_id(checkpoints)

checkspecs_path = client.checkspecs_path
specs_by_id: dict = {}
if checkspecs_path.exists():
    specs_by_id = load_checkspecs(checkspecs_path)
pending_media = sum(
    1
    for s in specs_by_id.values()
    if str(s.get("status_class") or "") == "pending"
    and str(s.get("data_source") or "") in ("report_images", "report_attachments")
)
print("checkpoints:", client.checkpoints_path.relative_to(PROJECT_ROOT))
print(
    "checkspecs:",
    checkspecs_path.relative_to(PROJECT_ROOT) if checkspecs_path.exists() else "(missing — unable media rows skipped)",
)
print("pending photo/attachment content specs:", pending_media)
print("gi rules:", client.gi_rules.relative_to(PROJECT_ROOT) if client.gi_rules.exists() else client.gi_rules)

if USE_FLAWED:
    report_paths = [p for p, _key in client.flawed_reports()]
elif str(REPORTS_DIR).strip():
    report_paths = list_report_jsons(resolve_path(REPORTS_DIR), REPORT_GLOB)
else:
    report_paths = client.corrected_reports()

if not report_paths:
    raise FileNotFoundError(
        "No reports selected — set REPORTS_DIR to a folder of *.json, "
        "or leave it empty for corrected/, or USE_FLAWED=True."
    )

print(f"{len(checkpoints)} checkpoints | {len(report_paths)} reports")
for p in report_paths:
    try:
        print(" -", p.relative_to(PROJECT_ROOT))
    except ValueError:
        print(" -", p)


rebuilt from data/clients/ribkoff/gi/rules.md → 29 checkpoints (format=harness) → 29 specs
checkpoints: data/pipeline/checkpoints/ribkoff_checkpoints.json
checkspecs: data/pipeline/checkspecs/ribkoff_checkspecs.json
pending photo/attachment content specs: 3
gi rules: data/clients/ribkoff/gi/rules.md
29 checkpoints | 7 reports
 - data/clients/ribkoff/flawed/Q2613916538_Jun29_flawed.json
 - data/clients/ribkoff/flawed/Q2614146161_flawed.json
 - data/clients/ribkoff/flawed/Q2614154465_Jun29_flawed.json
 - data/clients/ribkoff/flawed/Q2614157738_271904_flawed.json
 - data/clients/ribkoff/flawed/Q2614430288_flawed.json
 - data/clients/ribkoff/flawed/Q2614430689_flawed.json
 - data/clients/ribkoff/flawed/Q2614489409_flawed.json


In [3]:
orders = []  # (order_id, flags)
summaries = []
reports_by_order = {}  # for Original Content enrichment from live JSON

for report_path in report_paths:
    t0 = time.perf_counter()
    report = json.loads(report_path.read_text(encoding="utf-8"))
    # Production path (same as scripts/eval_all.py / scripts/run_review.py):
    # parse → compile/checkspecs cache → atoms → cascade → symbolic eval
    run = run_production_review(
        report_path,
        client.checkpoints_path,
        project_root=PROJECT_ROOT,
        enable_vision=ENABLE_VISION,
        extract_model=EXTRACT_MODEL,
        judge_model=EXTRACT_MODEL,
    )
    summary = format_cost_summary(run)
    elapsed = round(time.perf_counter() - t0, 1)
    order_id = order_id_for_report(report_path, report)
    flags = list(summary.get("flags") or [])
    orders.append((order_id, flags))
    reports_by_order[order_id] = report
    summaries.append(
        {
            "order_id": order_id,
            "report": report_path.name,
            "blocking": summary.get("blocking_flags_count"),
            "total_flags": summary.get("total_flags_count"),
            "unable": summary.get("unable_to_check_count"),
            "elapsed_s": elapsed,
            "cost_usd": (summary.get("cost_usd") or {}).get("total", {}).get("total_cost_usd"),
        }
    )
    dump = OUT_DIR / f"{order_id}_review.json"
    dump.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print(
        f"{order_id}: flags={len(flags)} blocking={summary.get('blocking_flags_count')} "
        f"unable={summary.get('unable_to_check_count')} {elapsed}s → {dump.name}"
    )

print("done", len(orders), "orders")


Q2613916538: flags=5 blocking=5 unable=8 2.3s → Q2613916538_review.json
Q2614146161: flags=5 blocking=5 unable=9 2.0s → Q2614146161_review.json
Q2614154465: flags=4 blocking=4 unable=8 3.7s → Q2614154465_review.json
Q2614157738: flags=5 blocking=5 unable=7 2.5s → Q2614157738_review.json
Q2614430288: flags=4 blocking=4 unable=8 1.7s → Q2614430288_review.json
Q2614430689: flags=6 blocking=6 unable=8 1.8s → Q2614430689_review.json
Q2614489409: flags=4 blocking=4 unable=10 1.8s → Q2614489409_review.json
done 7 orders


In [4]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_xlsx = OUT_DIR / f"{CLIENT}_findings_{stamp}.xlsx"
write_findings_workbook(
    orders,
    requirements,
    out_xlsx,
    summary_rows=summaries,
    meta_by_id=meta_by_id,
    reports_by_order=reports_by_order,
    specs_by_id=specs_by_id,
)
print("Wrote", out_xlsx)
print("Sheets: Findings | Summary | Verdict legend")
print(f"{sum(len(f) for _, f in orders)} violation flags across {len(orders)} orders")
print(f"+ pending media manual-check rows from {pending_media} specs (photo/attachment content only)")


Wrote /Users/williambrun/Documents/test_GI_reports/data/pipeline/findings/ribkoff_findings_20260721_173708.xlsx
Sheets: Findings | Summary | Verdict legend
33 violation flags across 7 orders
+ pending media manual-check rows from 3 specs (photo/attachment content only)
